In [1]:
import sys
import os
import multiprocessing

# CRITICAL: Set multiprocessing start method to 'spawn' BEFORE any CUDA initialization
# This fixes the "Cannot re-initialize CUDA in forked subprocess" error in Jupyter
multiprocessing.set_start_method('spawn', force=True)

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vllm_wrapper import VLLMAtomizationModel

# Import the original SAFE implementation
from third_party.factscore import atomic_facts
import itertools

def get_atomic_facts_safe(response: str, model, debug=False, query=None):
    """Wrapper that uses the original SAFE implementation with correct paths."""
    demon_dir = os.path.join(lff_root, "third_party", "factscore", "demos")
    atomic_fact_generator = atomic_facts.AtomicFactGenerator(
        api_key='', 
        demon_dir=demon_dir,
        gpt3_cache_file='', 
        other_lm=model,
        query=query
    )
    
    # Monkey patch the generate method to print prompts if debug=True
    if debug:
        original_generate = model.generate
        def debug_generate(prompt, **kwargs):
            print("="*80)
            print("PROMPT SENT TO MODEL:")
            print("="*80)
            print(prompt)
            print("="*80)
            result = original_generate(prompt, **kwargs)
            print("\nMODEL RESPONSE:")
            print("="*80)
            print(result)
            print("="*80)
            return result
        model.generate = debug_generate
    
    facts, _ = atomic_fact_generator.run(response)
    
    # Restore original generate if we patched it
    if debug:
        model.generate = original_generate
    
    # Convert to dict format
    facts_as_dict = [
        {'sentence': sentence, 'atomic_facts': identified_atomic_facts}
        for sentence, identified_atomic_facts in facts
    ]
    all_atomic_facts_list = list(
        itertools.chain.from_iterable([f['atomic_facts'] for f in facts_as_dict])
    )
    
    return {
        'num_claims': len(all_atomic_facts_list),
        'sentences_and_atomic_facts': facts,
        'all_atomic_facts': facts_as_dict,
    }


In [4]:
dummy_text_to_atomize = "He had an IQ of 160"
llm = VLLMAtomizationModel()
atomized = get_atomic_facts_safe(dummy_text_to_atomize, llm, debug=False, query="Tell me about Albert Einstein")

print(atomized)

{'num_claims': 1, 'sentences_and_atomic_facts': [('He had an IQ of 160', ['Albert Einstein had an IQ of 160.'])], 'all_atomic_facts': [{'sentence': 'He had an IQ of 160', 'atomic_facts': ['Albert Einstein had an IQ of 160.']}]}


In [2]:
import json
import os

responses = []
with open(os.getcwd() + "/data_for_git/responses.jsonl", "r") as f:
    for line in f:
        responses.append(json.loads(line))

# print one of the responses

def get_original_prompt(prompt):
    return prompt.split("Based on the documents above, i now want you to: ")[1].split(",")[0]

In [5]:
import threading

# Thread-safe list to collect results
results = []
results_lock = threading.Lock()

def worker(prompt, response, model):
    original_prompt = get_original_prompt(prompt)
    """Worker function that collects results"""
    result = get_atomic_facts_safe(response, model, debug=False, query=original_prompt)
    # print("original prompt", original_prompt)
    # print("prompt", prompt)
    with results_lock:
        results.append({"prompt": original_prompt, "response": response, "result": result})
threads = []
# Fix: enumerate returns (index, item), so iterate directly
for query in responses[:50]:
    for response in query["responses"]:
        t = threading.Thread(target=worker, args=(query["prompt"], response, llm))
        t.start()
        threads.append(t)

for t in threads:
    t.join()

# Now results contains all the atomic facts
print(f"Processed {len(results)} responses")
# Access results: results[0], results[1], etc.

Exception in thread Thread-35 (treat_prompt):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/root/IR_project/long_form_factuality/third_party/factscore/atomic_facts.py", line 265, in treat_prompt
    output = self.other_lm.generate(prompt_to_send)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/IR_project/actually_important/vllm_wrapper.py", line 68, in generate
    structured_output = client.chat.completions.create(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/venv/lib/python3.12/site-packages/openai/_utils/_utils.py", line 286, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/root/venv/lib/python3.12/site-packages/openai/resources/chat/completions/completions.py", line 1189, in create
    return self._post(
        

Processed 168 responses


In [5]:
import json
with open(os.getcwd() + "/data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")


In [6]:
total_number_of_facts = 0
total_number_of_sentences = 0
for result in results:
    total_number_of_facts += result['num_claims']
    total_number_of_sentences += len(result['sentences_and_atomic_facts'])

print(f"Total number of facts: {total_number_of_facts}")
print(f"Total number of sentences: {total_number_of_sentences}")
import random
# print a random sentence and its atomic facts
random_response = random.choice(results)
random_sentence = random.choice(random_response['sentences_and_atomic_facts'])
print(f"Random sentence: {random_sentence[0]}")
print(f"Atomic facts: {random_sentence[1]}")


KeyError: 'num_claims'

In [ ]:
with open("data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")